In [2]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt

from utils import voigt_profile

In [3]:
delta_wv = 0.069

wv = np.append(np.arange(-2,3), 4) * delta_wv
wv_fine = np.arange(-3, 4.1, 0.1) * delta_wv


sigma = 0.043
gamma = 0.053
contpos = -1
acc = 1e-3
lam = 1e-6

n_wv = len(wv)

wv_min = np.min(np.delete(wv, contpos) if contpos is not None else wv)
wv_max = np.max(np.delete(wv, contpos) if contpos is not None else wv)
wvc = (wv_min + wv_max) / 2
wv_ = np.arange(wv_min, wv_max + acc / 2, acc, dtype=np.float32)
n_wv_ = len(wv_)

xi = np.expand_dims(wv, axis=1) - np.expand_dims(wv_, axis=0)
A = voigt_profile(xi, sigma, gamma)
A0 = np.mean(A, axis=0, keepdims=True)
A -= A0

M = np.zeros((n_wv, n_wv))
N = np.zeros((n_wv, n_wv))

for k in range(n_wv_):
    gk = voigt_profile(wv_[k] - wvc, sigma, gamma) ** 2

    for j in range(n_wv):
        mjk = gk * (voigt_profile(wv_[k] - wv[j], sigma, gamma, modified=True) - A0[0,k])
        njk = gk * A[j,k]

        for i in range(n_wv):
            M[j, i] += mjk * A[i,k]
            N[j, i] += njk * A[i,k]

Q = M @ np.linalg.inv(N + lam * np.identity(n_wv))
Q = Q @ (np.identity(n_wv) - 1 / n_wv) + 1 / n_wv

In [4]:
Q

array([[ 6.48028145e-02,  1.72249933e-01,  2.02291288e-02,
         2.30974222e-02, -1.26528332e-01,  8.46149033e-01],
       [ 2.38856733e-01, -9.73752503e-02,  2.22002651e-01,
         2.11622363e-02, -9.74173781e-02,  7.12771008e-01],
       [ 3.10200784e-02,  2.19664656e-01, -1.02907571e-01,
         2.42828708e-01, -9.48637228e-02,  7.04257851e-01],
       [ 2.86539882e-02, -8.88906368e-04,  2.20782904e-01,
        -7.24959663e-02,  1.09886682e-01,  7.14061299e-01],
       [ 2.41102673e-02, -4.09910954e-03,  1.96735483e-02,
         2.00734642e-01, -8.71560894e-02,  8.46736742e-01],
       [ 1.96619443e-02, -1.67061262e-02,  8.98205420e-03,
         1.21324744e-03, -2.63443495e-02,  1.01319323e+00]])

In [6]:
offset = np.random.normal(0,0.5 * delta_wv)

f = voigt_profile(wv - offset, sigma, gamma)
f_ = Q @ f

f_fine = voigt_profile(wv_fine - offset, sigma, gamma, modified=False)
f_fine_ = voigt_profile(wv_fine - offset, sigma, gamma, modified=True)

plt.figure(figsize=(10,8))
plt.plot(wv, f, '.', ms=25)
plt.plot(wv, f_, '.', ms=25)
plt.plot(wv_fine, f_fine)
plt.plot(wv_fine, f_fine_)

plt.tight_layout()

In [8]:
x = np.arange(-1, 1, 1e-2)
#x = wv_fine

f_fine = voigt_profile(x, sigma, gamma, modified=False)
f_fine_ = voigt_profile(x, sigma, gamma, modified=True)

plt.figure(figsize=(10,8))
plt.plot(x, f_fine_ / f_fine)

In [1]:
from scipy.special import voigt_profile as vp

f_fine = voigt_profile(wv_fine, sigma, gamma, modified=False)
f_fine_ = vp(wv_fine, sigma, gamma)

plt.figure(figsize=(10,8))
plt.plot(wv_fine, f_fine / np.max(f_fine))
plt.plot(wv_fine, f_fine_ / np.max(f_fine_))

plt.tight_layout()

NameError: name 'voigt_profile' is not defined